# 01. 소리 듣기 + 스펙트로그램

**목적**: 1초 조각을 고르는 규칙을 정하기 전에 실제 소리를 듣고 눈으로 확인한다.

| 구역 | 확인할 것 |
|---|---|
| A | 클래스 × 데이터셋별 예시 (사이렌·경적·배경이 어떻게 다르게 보이는지) |
| B | 가장 긴 경적 파일 — 경적이 파일 전체에 있나, 몇 번뿐인가 |
| C | 두 데이터셋에서 라벨이 엇갈린 겹친 녹음 2건 |
| D | 16kHz 미만 파일 |
| E | 1초 조각 통계 — 파일 안에서 '큰 조각'이 얼마나 되나 |

> ⚠️ **커밋 전에 반드시 출력 지우기** (VS Code 위쪽 `…` → **Clear All Outputs**).
> 출력에는 AI Hub 소리가 그대로 들어가는데, AI Hub는 재배포 금지라 공개 레포에 올리면 안 된다.

그래프 안 글자는 영어로 쓰고(한글 글꼴 깨짐 방지), 한글 정보는 그래프 위에 글로 출력한다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from IPython.display import Audio, display

# 노트북 위치(notebooks/)나 프로젝트 폴더 어디서 열어도 동작하게
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
df = pd.read_csv(ROOT / "data/processed/manifest.csv", low_memory=False)

SR = 16000          # 학습에 쓸 샘플레이트
WIN, HOP = 1.0, 0.5 # 1초 조각, 0.5초 간격 (웹 알림과 같은 간격)
RNG = np.random.default_rng(0)

print(f"{len(df):,}행 · 학습 사용 {df['use'].sum():,}개")
df.groupby(["class", "dataset"]).size().unstack(fill_value=0)

In [ ]:
def load(path, sr=SR):
    """16kHz 모노로 읽기 (스테레오는 평균)"""
    y, _ = librosa.load(ROOT / path, sr=sr, mono=True)
    return y


def window_db(y, sr=SR, win=WIN, hop=HOP):
    """1초 조각마다 소리 크기(dB). 1초보다 짧으면 조각 1개"""
    w, h = int(win * sr), int(hop * sr)
    if len(y) < w:
        y = np.pad(y, (0, w - len(y)))
    starts = np.arange(0, len(y) - w + 1, h)
    rms = np.array([np.sqrt(np.mean(y[s:s + w] ** 2)) for s in starts])
    return starts / sr, 20 * np.log10(rms + 1e-10)


def show(row, play=True):
    """정보 출력 + 스펙트로그램 + 1초 조각 크기 그래프 + 재생"""
    y = load(row["path"])
    print(f"[{row['class']}] {row['dataset']} · {row['orig_label']} · {len(y)/SR:.1f}초 · split={row.get('split', '')}")
    print(f"   {row['path']}")

    fig, ax = plt.subplots(2, 1, figsize=(11, 4.2), sharex=True,
                           gridspec_kw={"height_ratios": [3, 1.3]})
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=400, hop_length=160, n_mels=64)
    librosa.display.specshow(librosa.power_to_db(mel, ref=np.max), sr=SR, hop_length=160,
                             x_axis="time", y_axis="mel", ax=ax[0])
    ax[0].set_title(f"{row['class']} | {row['dataset']} | {len(y)/SR:.1f}s")
    ax[0].set_xlabel("")

    t, db = window_db(y)
    if len(db) == 1:   # 1.25초보다 짧으면 조각이 1개 → 가로선으로 표시
        ax[1].hlines(db[0], 0, len(y) / SR, color="k")
    else:
        ax[1].step(t + WIN / 2, db, where="mid", color="k")
    ax[1].axhline(db.max() - 6, ls="--", c="tab:red", lw=1, label="max-6dB")
    ax[1].axhline(db.max() - 10, ls=":", c="tab:orange", lw=1, label="max-10dB")
    ax[1].set_ylabel("1s dB")
    ax[1].legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.show()
    if play:
        display(Audio(y, rate=SR))

## A. 클래스 × 데이터셋별 예시

각 칸에서 무작위 1개씩 (총 9개). 다른 예시를 보고 싶으면 맨 위 `RNG`의 숫자를 바꾼다.

**볼 것**: 사이렌은 높낮이가 오르내리는 줄무늬, 경적은 짧고 굵은 가로줄, 배경은 넓게 퍼진 소음.

In [ ]:
used = df[df["use"] & df["duration"].notna()]  # 읽을 수 있는 학습용 파일만
for (cls, ds), g in used.groupby(["class", "dataset"]):
    show(g.iloc[RNG.integers(len(g))])

## B. 가장 긴 경적 파일 3개 (AI Hub)

**볼 것**: 아래 그래프에서 빨간 점선(가장 큰 조각보다 6dB 작은 선) 위로 올라오는 조각이 몇 개인가?
→ 경적이 파일 전체에 있는지, 몇 번뿐인지 판단.

In [ ]:
long_horn = used[(used["class"] == "horn") & (used["dataset"] == "aihub")].nlargest(3, "duration")
for _, r in long_horn.iterrows():
    show(r)

## C. 두 데이터셋에서 클래스가 엇갈린 겹친 녹음

같은 Freesound 녹음인데 ESC-50과 UrbanSound8K에서 다른 클래스로 연결된 것.
UrbanSound8K는 긴 녹음을 구간별로 잘라 라벨을 붙이므로 **라벨이 틀린 게 아니라 다른 구간**일 수 있다.

**볼 것**: '배경'으로 들어간 쪽에 경적·사이렌이 섞여 있으면 → 그 파일은 제외(exclude) 검토.

In [ ]:
both = df[df["dataset"].isin(["esc50", "us8k"])]
cls_per_id = both.groupby("source_id")["class"].nunique()
ds_per_id = both.groupby("source_id")["dataset"].nunique()
conflict_ids = cls_per_id[(cls_per_id > 1) & (ds_per_id > 1)].index

conflict = both[both["source_id"].isin(conflict_ids)].sort_values(["source_id", "dataset"])
display(conflict[["source_id", "dataset", "orig_label", "class", "split", "duration", "path"]])

In [ ]:
for _, r in conflict.iterrows():
    show(r)

## D. 16kHz 미만 파일

16kHz로 올려도 원래 없던 높은 소리는 생기지 않는다. 어떤 클래스에 몇 개인지만 보고, 1개 들어본다.

In [ ]:
low = df[df["sample_rate"] < SR]
display(low.groupby(["class", "orig_label", "sample_rate"]).size().rename("개수").to_frame())
if len(low):
    show(low[low["use"]].iloc[0] if low["use"].any() else low.iloc[0])

## E. 1초 조각 통계

클래스 × 데이터셋마다 무작위 최대 150개 파일을 골라, 파일마다
- 1초 조각이 몇 개 나오는지
- 그중 **가장 큰 조각보다 6dB / 10dB 이내인 조각**이 몇 %인지

를 센다. 이 숫자로 "큰 조각만 고르기" 기준과 "파일당 조각 수 상한"을 정한다. (1~2분 걸림)

In [ ]:
N = 150
stats = []
for (cls, ds), g in used.groupby(["class", "dataset"]):
    for _, r in g.sample(min(N, len(g)), random_state=0).iterrows():
        _, db = window_db(load(r["path"]))
        stats.append({
            "class": cls, "dataset": ds, "windows": len(db),
            "loud6": np.mean(db >= db.max() - 6),
            "loud10": np.mean(db >= db.max() - 10),
        })
stats = pd.DataFrame(stats)

summary = stats.groupby(["class", "dataset"]).agg(
    파일=("windows", "size"),
    조각수_중앙값=("windows", "median"),
    조각수_최대=("windows", "max"),
    큰조각6dB_비율=("loud6", "median"),
    큰조각10dB_비율=("loud10", "median"),
).round(2)
summary

## 메모 (2026-11-18 확인 결과)

| 구역 | 확인 결과 |
|---|---|
| A | 클래스별 모양이 뚜렷이 다름. 사이렌 = 오르내리는 선, 경적 = 가로줄 여러 겹, 배경 = 넓게 퍼진 소음. **사이렌 패턴이 데이터셋(나라)마다 다름**: AI Hub(한국)는 부드러운 물결, ESC-50은 계단식으로 끊김, US8K는 소음 속 완만한 상승 (예시 몇 개 기준). ESC-50·US8K 경적은 소음보다 작아 희미함. ESC-50은 짧은 소리 뒤를 **디지털 무음(−200dB)**으로 채움 |
| B | AI Hub 가장 긴 경적 3개(58초대) 모두 **경적이 파일 전체에 있음** (실제 거리보다 비현실적으로 김 → 인위적 녹음). 차량경적 58.2초 파일은 13~19초에 경적이 아닌 큰 소리가 겹쳐 가장 큰 조각이 됨 → **"큰 조각만 고르기"는 AI Hub에서 경적 조각을 버리게 됨** |
| C | 녹음 fs_176638: US8K "engine_idling"(배경) 3개 중 **2개에 경적이 섞임** (`fold4/176638-5-0-1.wav`, `fold4/176638-5-0-2.wav`) → 수동 제외. US8K는 구간마다 가장 두드러진 소리 하나만 라벨로 붙여서 배경 라벨에 경적·사이렌이 섞일 수 있음 → YAMNet으로 배경 전체 검사 예정. 녹음 fs_43802: ESC 사이렌 + US8K 0.2초 경적 3개, 라벨 모두 맞음 |
| D | 16kHz 미만 58개 (US8K, 배경 49 · 사이렌 9). 11,025Hz 파일은 5.5kHz 위가 비어 있음. 전체의 0.15%라 그대로 사용 |
| E | 파일당 1초 조각 수 중앙값: AI Hub 배경 13 · 경적 4 · 사이렌 20 (최대 118), ESC-50 9, US8K 4~7. 가장 큰 조각보다 10dB 이내 비율: **ESC-50 경적 0.44**, 나머지 1.00 |

**→ 정한 조각 고르기 규칙**
1. 16kHz 모노, 1초 조각 · 0.5초 간격, 1초 미만 파일은 1초로 채움
2. 거의 무음(−60dB 미만) 조각 버림
3. ESC-50 · US8K의 사이렌 · 경적만 가장 큰 조각보다 10dB 넘게 작은 조각 버림 (AI Hub와 배경은 크기 기준 없음)
4. 학습(train)은 파일당 최대 10조각, 파일 전체에서 고르게. 검증 · 테스트는 줄이지 않음 (파일 단위로 알림 규칙 평가)
5. 수동 제외 목록 적용